# Research Question 3
Is there a threshold at which cooperative subgraph transitions from fragmented components to a giant cooperative component?
## Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm
from Prison import (
    ImitationStrategy,
    NetworkSimulation,
    experiment,
    TitForTatStrategy,
    ReinforcementLearningStrategy,
)
from pathlib import Path

In [ ]:
%matplotlib inline
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["animation.embed_limit"] = 500

In [ ]:
directory = "Plots/Results_RQ3/"
Path(directory).mkdir(parents=True, exist_ok=True)

## Fragmentation Analysis

In [ ]:
class FragmentationAnalysis:
    def __init__(self, STRATEGY, N_SIDE, MAX_STEPS, TRIALS, BETAS):
        self.strategy = STRATEGY
        self.n_side = N_SIDE
        self.steps = MAX_STEPS
        self.trials = TRIALS
        self.betas = BETAS

    def simulation(self):
        """
        Runs a simulation with the specified parameters until the simulation reaches a stable attractor state.
        """
        print(
            f"Starting Simulation for: {self.n_side}x{self.n_side} Grid, {self.trials} Trials per Beta..."
        )
        results = []

        for beta in tqdm(self.betas):
            pm = {
                ("C", "C"): (1.0, 1.0),
                ("C", "D"): (0.0, beta),
                ("D", "C"): (beta, 0.0),
                ("D", "D"): (0, 0),
            }

            S_max_list = []
            S_mean_list = []
            N_clusters_list = []

            for trial_id in range(self.trials):
                sim = NetworkSimulation(
                    kind="grid",
                    rounds=self.steps,
                    n=self.n_side * self.n_side,
                    payoff_matrix=pm,
                    strategy=self.strategy,
                    strategy_kwargs={},
                    history_window=10,
                    store_history=True,
                    store_snapshots=False,
                    seed=int(trial_id * beta + 42),
                )

                sim.run_until_attractor(
                    max_steps=self.steps, check_every=25, store_cycle_states=False
                )

                S_max, S_mean, N_clusters = self.calculate_metrics(sim)
                S_mean_list.append(S_mean)
                S_max_list.append(S_max)
                N_clusters_list.append(N_clusters)

            results.append(
                {
                    "beta": beta,
                    "S_max_mean": np.mean(S_max_list),
                    "S_max_std": np.std(S_max_list),
                    "S_mean": np.mean(S_mean_list),
                    "S_mean_std": np.std(S_mean_list),
                    "N_clusters": np.mean(N_clusters_list),
                    "N_clusters_std": np.std(N_clusters_list),
                }
            )

        self.df = pd.DataFrame(results)

    def calculate_metrics(self, simulation):
        """
        Calculates the cooperation metrics at the end of the simulation .
        S_max: Number of nodes in the largest cluster of cooperators.
        S_mean: Mean number of nodes in clusters of cooperators.
        N_clusters: Number of clusters of cooperators.
        """

        state = simulation._get_state()
        coop_nodes = [n for n, action in state.items() if action == 0]

        if not coop_nodes:
            return 0.0, 0.0, 0.0

        G_coop = simulation.graph.subgraph(coop_nodes)
        components = list(nx.connected_components(G_coop))

        if not components:
            return 0.0, 0.0, 0.0

        sizes = sorted([len(c) for c in components], reverse=True)

        N_clusters = len(sizes)
        S_mean = np.mean(sizes)
        S_max = sizes[0]

        return S_max, S_mean, N_clusters

    def visualize(self, save=False, filename=None, lines={}):
        """
        Generates a visualization of the FragmentationAnalysis showing S_max, S_mean and N_clusters over a changing beta.
        """
        fig, ax1 = plt.subplots(figsize=(12, 7))

        colors = {
            "S_max": "tab:blue",
            "S_mean": "tab:red",
            "N_clusters": "tab:green",
        }

        ax1.set_xlabel(r"Temptation ($\beta$)", fontsize=14)
        ax1.set_ylabel(r"Cooperative Cluster Size ($S$)", fontsize=14)
        ax1.grid(True, which="both", alpha=0.3)

        ax1.plot(
            self.df["beta"],
            self.df["S_max_mean"],
            color=colors["S_max"],
            marker="o",
            lw=2,
            label=r"$S_\text{max}$",
        )
        ax1.fill_between(
            self.df["beta"],
            self.df["S_max_mean"] - self.df["S_max_std"],
            self.df["S_max_mean"] + self.df["S_max_std"],
            color=colors["S_max"],
            alpha=0.2,
        )

        ax1.plot(
            self.df["beta"],
            self.df["S_mean"],
            color=colors["S_mean"],
            marker="o",
            lw=2,
            label=r"$\bar S$",
        )
        ax1.fill_between(
            self.df["beta"],
            self.df["S_mean"] - self.df["S_mean_std"],
            self.df["S_mean"] + self.df["S_mean_std"],
            color=colors["S_mean"],
            alpha=0.1,
        )

        ax2 = ax1.twinx()
        ax2.set_ylabel(r"Number of clusters ($N$)", fontsize=14)

        ax2.plot(
            self.df["beta"],
            self.df["N_clusters"],
            color=colors["N_clusters"],
            marker="o",
            linestyle="--",
            lw=2,
            label=r"$N_\text{clusters}$",
        )
        ax2.fill_between(
            self.df["beta"],
            self.df["N_clusters"] - self.df["N_clusters_std"],
            self.df["N_clusters"] + self.df["N_clusters_std"],
            color=colors["N_clusters"],
            alpha=0.1,
        )

        for x, extra_args in lines.items():
            style = {"color": "k", "linestyle": "dotted"}
            if extra_args:
                style.update(extra_args)
            ax1.axvline(x=x, **style)

        plt.title(
            f"Fragmentation of Cooperative Clusters as Temptation Increases\nGrid ({self.n_side}x{self.n_side}), {self.strategy.__name__}",
            fontsize=16,
        )
        fig.legend(
            loc="lower center",
            bbox_to_anchor=(0.5, -0.06),
            ncol=3,
            fontsize=12,
        )
        fig.tight_layout()
        if save:
            plt.savefig(f"{directory}{filename}.jpg", bbox_inches="tight")
        plt.show()

In [ ]:
imitation = FragmentationAnalysis(
    STRATEGY=ImitationStrategy,
    N_SIDE=20,
    MAX_STEPS=1000,
    TRIALS=30,
    BETAS=np.linspace(0.0, 3.0, 31),
)
imitation.simulation()

In [ ]:
my_lines = {0.6: {}, 1.6: {"alpha": 0.5}}
imitation.visualize(save=True, filename="Imitation_Fragmentation", lines={0.6: {}})

In [ ]:
rl = FragmentationAnalysis(
    STRATEGY=ReinforcementLearningStrategy,
    N_SIDE=20,
    MAX_STEPS=1000,
    TRIALS=3,
    BETAS=np.linspace(0.0, 3.0, 31),
)
rl.simulation()

In [ ]:
rl.visualize()

## Animation

In [ ]:
def animate(STRATEGY, STEPS, N_SIDE, BETAS, SAVE=False):
    for beta in tqdm(BETAS):
        anim_path = f"{STRATEGY.__name__}_beta_{beta}.gif"

        pm = {
            ("C", "C"): (1.0, 1.0),
            ("C", "D"): (0.0, beta),
            ("D", "C"): (beta, 0.0),
            ("D", "D"): (0, 0),
        }

        gif = experiment(
            NetworkSimulation,
            strategy_class=STRATEGY,
            strategy_kwargs={},
            steps=STEPS,
            seed=int(beta + 42),
            interval=75,
            payoff_matrix=pm,
            title=rf"Cooperation over Time ($\beta$ = {beta:.2f})",
            kind="grid",
            n=N_SIDE * N_SIDE,
            is_grid=True,
            save_gif=SAVE,
            gif_path=f"{directory}{anim_path}",
        )

        if SAVE:
            print(f"Animation saved to {directory}{anim_path}")
        display(gif)

In [ ]:
animate(
    STRATEGY=ImitationStrategy,
    STEPS=100,
    N_SIDE=20,
    BETAS=[0.1, 0.5, 1.0, 2.0],
    SAVE=True,
)